## Initializing a quantum device

Each `Device` in QoolQit wraps a [Pulser](https://pulser.readthedocs.io/en/stable/tutorials/virtual_devices.html#) device and defines the hardware characteristics that the program will be compiled to and later executed on.

In [ ]:
from qoolqit import AnalogDevice, AnalogDeviceWithDMM, MockDevice

# An example of a mock device with no hardware constraints
device_ideal = MockDevice()

# An example of a real device
device_real = AnalogDevice()

# An example of a real device with a DMM channel.
device_real_digital = AnalogDeviceWithDMM()

The updated list of available devices is given below:

In [ ]:
from qoolqit import available_default_devices

available_default_devices()

Besides available default devices, relevant for QPU emulation, new QPU devices can be:
- imported remotely
- created from custom Pulser devices

#### Fetching a QoolQit device from a connection
Depending on your provider you might have different QPUs available to launch your quantum program to.
The list of available ones can be fetched through the specific connection handler object, with the generic `connection.fetch_available_devices()` method.

For the Pasqal Cloud service, for example, creating a QoolQit device from a connection object, simply reads as:

In [ ]:
from pulser_pasqal import PasqalCloud

from qoolqit import Device

connection = PasqalCloud()
print(connection.fetch_available_devices())

# fetch QoolQit device
fresnel_device = Device.from_connection(connection=connection, name="FRESNEL")
print(fresnel_device)

On a QRMI-enabled cluster, you can pass `PulserQRMIConnection()` as the connection and submit the same script with `sbatch --qpu=<backend> ...` (for example `FRESNEL` or `EMU_FREE`).

For a Qaptiva managed cluster, you can similarly pass `PulserQLMConnection()`.

### Create a QoolQit device from a Pulser device
A custom QoolQit device can also be built straight from any Pulser device, with any desired specification.
Please, refer to [Pulser documentation](https://docs.pasqal.com/pulser/tutorials/virtual_devices/) to learn how to make a custom device.

In [ ]:
from dataclasses import replace

from pulser import AnalogDevice as PulserAnalogDevice

from qoolqit import Device

# Converting the pulser Device object into a VirtualDevice object
VirtualAnalog = PulserAnalogDevice.to_virtual()
# Replacing desired values
ModdedAnalogDevice = replace(VirtualAnalog, max_radial_distance=100, max_sequence_duration=7000)

# Wrap a Pulser device object into a QoolQit Device
mod_analog_device = Device(pulser_device=ModdedAnalogDevice)
print(mod_analog_device)

## Compilation

Once a `QuantumProgram` is defined and a `Device` is selected one can proceed with the compilation by means of the method `compile_to`. This method will execute what has been discussed in the [introduction](./rationale.md) mapping adimensional parameters to physical quantities according to specific default rules.

In [ ]:
from qoolqit import AnalogDevice, Drive, PiecewiseLinearWaveform, QuantumProgram, Register

# Defining the Drive
wf0 = PiecewiseLinearWaveform([1.0, 2.0, 1.0], [0.0, 0.5, 0.5, 0.0])
wf1 = PiecewiseLinearWaveform([1.0, 2.0, 1.0], [-1.0, -1.0, 1.0, 1.0])
drive = Drive(amplitude = wf0, detuning = wf1)

# Defining the Register
coords = [(0.0, 0.0), (0.0, 1.0), (1.0, 0.0), (1.0, 1.0)]
register = Register.from_coordinates(coords)

# Creating the Program
program = QuantumProgram(register, drive)
device = AnalogDevice()

# Compilation to AnalogDevice
program.compile_to(device)
print(program)

Now that the program has been compiled, we can inspect the compiled sequence, which is an instance of a Pulser `Sequence`.

In [ ]:
pulser_sequence = program.compiled_sequence

Finally, we can draw both the original program and the compiled sequence.

In [ ]:
program.draw()
program.draw(compiled = True)

## Special compilation: maximum allowed duration

Whenever possible, QoolQit compilation directly maps the input quantum program to hardware instructions while preserving energy-to-time ratios (see [compilation rationale](./rationale.md)). In some applications, however, it is advantageous to decouple time from compilation. For example, in adiabatic protocols, users may wish to set the program duration independently, such as using the longest duration supported by the hardware.

<div class="admonition warning">
<p class="admonition-title">Warning</p>
<p>Programs compiled with the flag <code>device_max_duration_ratio</code> are not portable across different devices.</p>
</div>

To do so, set the corresponding flag at compilation:

In [ ]:
# Compilation to AnalogDevice max duration
program.compile_to(device, device_max_duration_ratio=1.0)
program.draw(compiled = True)

As anticipated, this flag will decouple time from the compilation, letting the user set a time relative to the maximum allowed by the selected device.
Moreover, as a drive's amplitude/detuning can be composed by many waveforms, this feature will rescale them preserving their relative durations.